# Course 27 lab — Testing and quality for agentic SDD

Build an independent, specification-traceable quality decision for Northstar's AI-2700 classification change. The notebook is credential-free and imports the same control plane verified by repository tests.

**Safety boundary:** candidate tests provide feedback; only authenticated independent evidence can satisfy quality obligations. No notebook result authorizes release.

## 1. Environment and mental model

The deterministic fixture uses an exact subject, a governed requirement/test-kind map, independent oracles, four critical mutations, and three failure experiments. Observe decision codes and evidence coverage rather than hidden reasoning.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from quality_app.control_plane import QualityControlPlane
from quality_app.simulation import NOW, evidence_for, run_evaluation, run_scenario, subject_for

plane = QualityControlPlane(ROOT)
subject = subject_for(plane)
subject.payload()


## 2. Baseline: candidate-only evidence

First show why a passing candidate report is not an attestation. The control plane keeps it as diagnostic information and reports missing independent obligations.

In [ ]:
candidate = evidence_for(plane, subject, producer="candidate-agent")
candidate_report = plane.evaluate(subject, [candidate], NOW)
assert candidate_report.decision == "INSUFFICIENT_EVIDENCE"
[finding.code for finding in candidate_report.findings[:5]]


## 3. Independent portfolio

Now evaluate authenticated evidence with requirement-linked unit, property, contract, integration, negative, failure, and recovery cases. The exact policy-selected mutation and failure populations must also pass.

In [ ]:
independent = evidence_for(plane, subject)
accepted = plane.evaluate(subject, [independent], NOW)
assert accepted.decision == "QUALITY_EVIDENCE_ACCEPTED"
assert accepted.release_authorized is False
accepted.payload()


## 4. Failure injection: surviving authorization mutant

A green suite is weak if removing authorization does not make it fail. The policy-selected mutation exposes that blind spot and blocks the change.

In [ ]:
mutation_failure = run_scenario(ROOT, "surviving-mutant")
assert mutation_failure["decision"] == "BLOCKED"
mutation_failure


## 5. Reliability experiment: flaky evidence

Mixed outcomes for the same subject cannot satisfy a quality obligation. Retry-until-green would hide uncertainty, so the result routes to accountable review.

In [ ]:
flaky = run_scenario(ROOT, "flaky")
assert flaky["decision"] == "REVIEW_REQUIRED"
flaky


## 6. Evaluation with explicit populations

The labelled evaluation runs twelve scenarios twice. The metrics retain numerator, denominator, and the narrow claim boundary. They are fixture conformance results, not production quality benchmarks.

In [ ]:
evaluation = run_evaluation(ROOT)
assert evaluation["metrics"]["exact_decision_accuracy"]["denominator"] == 24
assert evaluation["metrics"]["exact_decision_accuracy"]["value"] == 1.0
evaluation["metrics"]


## 7. Compare the alternatives

| Design | Speed | Independence | Best use |
|---|---:|---:|---|
| Candidate tests only | High | Low | Inner-loop feedback |
| Independent acceptance tests | Medium | High for specified behavior | Protected merge evidence |
| Mutation + failure portfolio | Lower | Higher fault sensitivity | Consequential boundaries |
| Runtime verification | Later | Measures deployed behavior | SLOs, drift, rare conditions |

Use the least costly portfolio that credibly addresses the consequence and uncertainty of the change.

## 8. Production upgrade and exercises

Production needs workload identity, isolated immutable workers, protected test catalogs, approved datasets, trusted time, replay protection, retention, cost budgets, flake ownership, and release-system integration.

Exercises:

1. Add a boundary case without weakening an oracle.
2. Design a mutation for an unsafe fallback.
3. Add an unknown-outcome timeout experiment with reconciliation.
4. Explain why QUALITY_EVIDENCE_ACCEPTED still leaves release_authorized false.
5. Complete the workshop starter and compare it with the reference strategy.